# 📘 CH16. NLP와 LLM 기초 — 강의용 노트북

> 🔄 **오늘의 컨셉**: 여러분은 오늘 엔터테인먼트 회사 **스텔라 엔터** 데이터팀에 인턴으로 합류했습니다. 소속 아이돌 그룹 **NEBULA(네뷸라)** 의 신곡 *Supernova* 가 방금 발매되었고, 전 세계에서 팬 댓글이 쏟아지는 중입니다. 팀장이 하루 동안 다섯 개의 미션을 차례로 줍니다.
>
> ① 쏟아지는 팬 댓글, 긍정일까 부정일까? (NLP 첫 만남) → ② 댓글을 AI가 읽는 조각으로 (토큰화) → ③ 단어에게 좌표를 주다 (임베딩) → ④ 번역·생성 AI의 엔진 열어보기 (Transformer와 GPT) → ⑤ 진짜 모델로 교체 배치 (HuggingFace Pipeline)

**구성** — 정규 4시간 (휴식 별도)

| 유닛 | 주제 | 예상 시간 |
|---|---|---|
| CH16-01 | AI는 어떻게 '말'을 읽을까 — NLP 첫 만남 | 40분 |
| CH16-02 | 토큰화 — 문장을 AI의 조각으로 | 45분 |
| CH16-03 | 임베딩 — 단어에게 좌표를 주다 | 50분 |
| CH16-04 | Transformer와 GPT의 속마음 | 45분 |
| CH16-05 | HuggingFace Pipeline — 3줄로 진짜 AI 쓰기 + 최종 대결 | 50분 |

> 📝 `💻 실습(TODO)`과 `📝 실습 과제`는 정규 수업이 아닌 **수업 후 자습 시간**에 진행합니다. 정규 수업은 개념 설명 · 시연 · Check Point 위주로 진행해주세요.
>
> 🆓 이 노트북은 유료 API를 전혀 사용하지 않습니다. 모든 모델은 무료 공개 모델이며 API 키가 필요 없습니다.
>
> ⚠️ **수업 전날 필수**: 바로 아래의 [사전 다운로드] 셀을 미리 실행해두세요. 모델 파일(약 1.5GB)을 내려받아 캐시에 저장합니다. 수업 당일에는 인터넷이 느려도 문제없이 진행됩니다.

## 🔧 사전 다운로드 — 수업 전날 반드시 실행!

아래 셀은 오늘 사용할 라이브러리와 모델을 미리 내려받는 셀입니다. **한 번만 실행하면 캐시에 저장**되어 이후에는 다운로드 없이 즉시 로딩됩니다.

- 설치: `transformers`, `torch`, `nltk`, `gensim`, `sentencepiece`
- 모델: 감성 분석(DistilBERT), 토크나이저(BERT), 텍스트 생성(GPT-2), 한→영 번역(opus-mt), 단어 임베딩(GloVe 50차원)

In [ ]:
# ============================================================
# NLP·LLM 수업 환경 설치
# 새 런타임에서 가장 먼저 실행
# ============================================================

# torch는 코랩 기본 버전을 그대로 사용합니다.
# 기존 Transformers 5.x 관련 파일만 완전히 교체합니다.
# 약 1분정도 샐 후 자동 세션 다운됩니다.

%pip uninstall -y transformers tokenizers huggingface-hub

%pip install -q --no-cache-dir --no-deps \
    "transformers==4.57.6" \
    "tokenizers==0.22.1" \
    "huggingface-hub==0.36.0"

%pip install -q --no-cache-dir \
    "sentencepiece==0.2.1" \
    "nltk==3.9.1" \
    "gensim==4.4.0"

print("✅ 설치 완료")
print("🔄 런타임을 자동으로 다시 시작합니다.")

# 설치 이전에 로드된 라이브러리가 남지 않도록 강제 재시작
import os
os.kill(os.getpid(), 9)

In [ ]:
# ============================================================
# NLP·LLM 수업용 데이터 및 모델 사전 다운로드
# (약 4 ~ 6분 진행)
# ============================================================

import os
import gc

# PyTorch만 사용
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ------------------------------------------------------------
# 1. 설치 상태 확인
# ------------------------------------------------------------

import torch
import transformers
import nltk
import gensim

from transformers import pipeline, AutoTokenizer
from transformers.pipelines import get_supported_tasks

print("PyTorch 버전:", torch.__version__)
print("Transformers 버전:", transformers.__version__)
print("Transformers 위치:", transformers.__file__)
print("Gensim 버전:", gensim.__version__)

# 잘못된 버전이면 다운로드를 시작하지 않고 즉시 중단
if transformers.__version__ != "4.57.6":
    raise RuntimeError(
        f"Transformers 버전 오류: {transformers.__version__}\n"
        "런타임을 연결 해제 및 삭제한 뒤 1번 셀부터 다시 실행하세요."
    )

supported_tasks = get_supported_tasks()

if "translation" not in supported_tasks:
    raise RuntimeError(
        "translation 파이프라인이 없습니다.\n"
        "Transformers가 정상적으로 교체되지 않았습니다."
    )

print("✅ translation 파이프라인 확인 완료")


# ------------------------------------------------------------
# 2. NLTK 데이터 다운로드
# ------------------------------------------------------------

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)

print("✅ NLTK 데이터 다운로드 완료")


# ------------------------------------------------------------
# 3. Hugging Face 모델 다운로드 및 캐시
# ------------------------------------------------------------

# BERT 토크나이저
tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-uncased"
)
del tokenizer
gc.collect()

print("✅ BERT 토크나이저 완료")


# 감성 분석 모델
model_pipeline = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    framework="pt",
    device=-1
)
del model_pipeline
gc.collect()

print("✅ 감성 분석 모델 완료")


# GPT-2 텍스트 생성 모델
model_pipeline = pipeline(
    "text-generation",
    model="gpt2",
    framework="pt",
    device=-1
)
del model_pipeline
gc.collect()

print("✅ GPT-2 모델 완료")


# 한국어 → 영어 번역 모델
model_pipeline = pipeline(
    "translation",
    model="Helsinki-NLP/opus-mt-ko-en",
    framework="pt",
    device=-1
)
del model_pipeline
gc.collect()

print("✅ 번역 모델 완료")


# ------------------------------------------------------------
# 4. GloVe 단어 임베딩 다운로드
# ------------------------------------------------------------

import gensim.downloader as api

glove = api.load("glove-wiki-gigaword-50")

del glove
gc.collect()

print("✅ GloVe 다운로드 완료")


# ------------------------------------------------------------
# 최종 확인
# ------------------------------------------------------------

print("\n" + "=" * 55)
print("✅ 사전 다운로드 완료! NLP·LLM 수업 준비 끝.")
print("✅ sentiment-analysis")
print("✅ text-generation")
print("✅ translation")
print("✅ GloVe")
print("=" * 55)

---
# 📘 CH16-01. AI는 어떻게 '말'을 읽을까 — NLP 첫 만남

⏱️ **예상 소요시간**: 40분

> 🎬 **미션 ①** — 팀장: "신곡 나온 지 3시간 만에 댓글이 10만 개예요. 팬들 반응이 좋은지 나쁜지, 사람이 다 읽지 말고 **컴퓨터가 판단하게** 만들어봐요."


## 🤔 먼저 생각해보기

- 댓글 10만 개를 사람이 1개당 3초씩 읽으면 얼마나 걸릴까요? (→ 약 83시간, 쉬지 않고 3일 반)
- 여러분은 "이 노래 미쳤다ㅋㅋ"가 칭찬인 걸 어떻게 아나요? 컴퓨터도 알 수 있을까요?
- 유튜브 자동 번역, 스팸 필터, 키보드 자동완성… 이런 기능들의 공통점은 뭘까요?

## 📖 핵심 개념

### 1) NLP(Natural Language Processing)란?

**자연어 처리** = 사람의 언어(자연어)를 컴퓨터가 이해하고 생성하도록 만드는 기술입니다.

이미 여러분의 일상 곳곳에 있습니다.

| 서비스 | NLP 태스크 |
|---|---|
| 파파고 / 구글 번역 | 번역 (Translation) |
| 스팸 메일 자동 분류 | 텍스트 분류 (Classification) |
| 뉴스 세 줄 요약 | 요약 (Summarization) |
| ChatGPT / Claude | 텍스트 생성 (Generation) |
| "시리야, 내일 날씨 알려줘" | 질의응답 (QA) + 음성 인식 |
| 쇼핑몰 리뷰 별점 예측 | **감성 분석 (Sentiment Analysis)** ← 오늘의 미션! |

### 2) 미션 데이터 공개 — NEBULA 신곡 팬 댓글

실제 서비스에서 수집되는 댓글은 이렇게 생겼습니다. 한국어·영어가 섞여 있고, 이모지와 신조어가 난무합니다.

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
import pandas as pd

# NEBULA 신곡 'Supernova' 발매 직후 수집된 팬 댓글 (실습용 가상 데이터)
fan_comments = [
    "이 노래 진짜 미쳤다ㅋㅋㅋ 벌써 100번 들음",
    "This song is a masterpiece, the chorus is stuck in my head!",
    "솔직히 이번 앨범은 좀 실망이야... 전작이 더 좋았어",
    "The music video is absolutely stunning, best comeback ever",
    "타이틀곡 별로다. 수록곡이 더 나은 듯",
    "NEBULA never disappoints, streaming all day!",
    "뮤비 미장센 무엇... 소름 돋았어 진심",
    "Honestly the autotune ruined it for me, disappointed",
    "스밍 준비 완료!! 오늘부터 1일 1스트리밍 간다",
    "I was not expecting much but wow, this exceeded everything",
    "노래는 좋은데 안무가 너무 아쉽다",
    "Worst title track they've ever released, skip",
    "기대 안 했는데 웬걸, 역대급이잖아?",
    "The bridge part gave me chills, pure art",
    "이게 노래냐... 3분이 아깝다",
    "Not bad at all, actually really impressive!",
    "멜로디 중독성 무엇 하루종일 흥얼거림",
    "Boring and generic, sounds like every other song",
    "컨셉 소화력 하나는 인정. 근데 노래는 글쎄",
    "Perfect song does not exi— oh wait, it just dropped",
]

df = pd.DataFrame({"comment": fan_comments})
print(f"수집된 댓글: {len(df)}개")
df.head(10)
```

</details>

In [ ]:
import pandas as pd

# NEBULA 신곡 'Supernova' 발매 직후 수집된 팬 댓글 (실습용 가상 데이터)
fan_comments = [
    "이 노래 진짜 미쳤다ㅋㅋㅋ 벌써 100번 들음",
    "This song is a masterpiece, the chorus is stuck in my head!",
    "솔직히 이번 앨범은 좀 실망이야... 전작이 더 좋았어",
    "The music video is absolutely stunning, best comeback ever",
    "타이틀곡 별로다. 수록곡이 더 나은 듯",
    "NEBULA never disappoints, streaming all day!",
    "뮤비 미장센 무엇... 소름 돋았어 진심",
    "Honestly the autotune ruined it for me, disappointed",
    "스밍 준비 완료!! 오늘부터 1일 1스트리밍 간다",
    "I was not expecting much but wow, this exceeded everything",
    "노래는 좋은데 안무가 너무 아쉽다",
    "Worst title track they've ever released, skip",
    "기대 안 했는데 웬걸, 역대급이잖아?",
    "The bridge part gave me chills, pure art",
    "이게 노래냐... 3분이 아깝다",
    "Not bad at all, actually really impressive!",
    "멜로디 중독성 무엇 하루종일 흥얼거림",
    "Boring and generic, sounds like every other song",
    "컨셉 소화력 하나는 인정. 근데 노래는 글쎄",
    "Perfect song does not exi— oh wait, it just dropped",
]

df = pd.DataFrame({"comment": fan_comments})
print(f"수집된 댓글: {len(df)}개")
df.head(10)

### 3) 컴퓨터의 눈에는 글자도 숫자다

컴퓨터는 '글자'라는 것을 모릅니다. 모든 문자는 내부적으로 **숫자(코드)** 로 저장됩니다.
- ord(): 문자 1개를 유니코드 숫자로 바꾸는 함수
- chr(): 반대로 숫자를 문자로 바꿀때 사용

In [ ]:
# 코드 예시





<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
text = "NEBULA 최고"

# 컴퓨터가 실제로 저장하는 값: 문자 하나하나가 전부 숫자
for ch in text:
    print(f"'{ch}' → {ord(ch)}")

# 즉, 컴퓨터 입장에서 이 문장은 그냥 숫자 나열일 뿐
print("\n컴퓨터가 보는 문장:", [ord(ch) for ch in text])
```

</details>

숫자 78(N), 69(E)… 이 나열만 보고 "긍정적인 댓글이네!"라고 판단할 수 있을까요? 불가능합니다.

그래서 NLP의 핵심 과제는 딱 하나입니다:

> **"글자 나열"을 어떻게 "의미를 담은 숫자"로 바꿀 것인가?**

오늘 하루 전체가 이 질문에 대한 답을 단계적으로 찾아가는 여정입니다. 가장 원시적인 방법부터 시작해봅시다.

### 4) 감성 분석기 v1 — 규칙 기반으로 직접 만들기

가장 단순한 아이디어: **긍정 단어 사전과 부정 단어 사전을 만들어서, 어느 쪽 단어가 많이 나오는지 세기.**

딥러닝 이전 시대(~2010년대 초)에 실제로 쓰이던 방식입니다.

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 감성 분석기 v1: 규칙(단어 사전) 기반
positive_words = ["미쳤다", "최고", "역대급", "소름", "중독성", "인정", "좋은데",
                  "masterpiece", "stunning", "best", "perfect", "impressive",
                  "art", "chills", "wow"]
negative_words = ["실망", "별로", "아쉽다", "아깝다", "글쎄",
                  "disappointed", "worst", "boring", "generic", "ruined", "skip"]

def sentiment_v1(comment):
    text = comment.lower()
    pos = sum(1 for w in positive_words if w in text)
    neg = sum(1 for w in negative_words if w in text)
    if pos > neg:
        return "긍정 😊"
    elif neg > pos:
        return "부정 😞"
    else:
        return "중립 😐"

# 댓글 3개로 테스트
for c in fan_comments[:3]:
    print(f"[{sentiment_v1(c)}] {c}")
```

</details>

In [ ]:
# 감성 분석기 v1: 규칙(단어 사전) 기반
positive_words = ["미쳤다", "최고", "역대급", "소름", "중독성", "인정", "좋은데",
                  "masterpiece", "stunning", "best", "perfect", "impressive",
                  "art", "chills", "wow"]
negative_words = ["실망", "별로", "아쉽다", "아깝다", "글쎄",
                  "disappointed", "worst", "boring", "generic", "ruined", "skip"]








# 댓글 3개로 테스트






<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 전체 20개 댓글에 일괄 적용 → 팀장에게 보고할 첫 리포트!
df["v1_판정"] = df["comment"].apply(sentiment_v1)

print("=== 감성 분석기 v1 리포트 ===")
print(df["v1_판정"].value_counts())
df
```

</details>

In [ ]:
# 전체 20개 댓글에 일괄 적용 → 팀장에게 보고할 첫 리포트!






### 5) v1의 한계 — 언어는 생각보다 교활하다

리포트를 자세히 보면 이상한 판정들이 보입니다. v1이 무너지는 대표 사례 세 가지:

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# v1을 무너뜨리는 함정 문장들
trap_sentences = [
    "Not bad at all, actually really impressive!",   # 부정어 뒤집기: not + bad = 긍정
    "기대 안 했는데 웬걸, 역대급이잖아?",              # 반전 구조
    "이게 노래냐... 3분이 아깝다",                     # '노래냐'는 사전에 없음 → 놓침
    "스밍 준비 완료!! 오늘부터 1일 1스트리밍 간다",     # 신조어(스밍) → 강한 긍정인데 중립 처리
]

for s in trap_sentences:
    print(f"[{sentiment_v1(s)}] {s}")
```

</details>

In [ ]:
# v1을 무너뜨리는 함정 문장들
trap_sentences = [
    "Not bad at all, actually really impressive!",   # 부정어 뒤집기: not + bad = 긍정
    "기대 안 했는데 웬걸, 역대급이잖아?",              # 반전 구조
    "이게 노래냐... 3분이 아깝다",                     # '노래냐'는 사전에 없음 → 놓침
    "스밍 준비 완료!! 오늘부터 1일 1스트리밍 간다",     # 신조어(스밍) → 강한 긍정인데 중립 처리
]







**v1이 실패하는 이유:**

1. **부정어(negation)**: "not bad"는 긍정이지만, v1은 'bad'만 보고 부정 판정
2. **반어법·맥락**: "이게 노래냐"는 단어 하나하나로는 중립이지만 전체 맥락은 강한 부정
3. **신조어·은어**: "스밍(스트리밍)", "1일 1스트리밍" 같은 팬덤 용어는 사전에 넣기 전까진 영원히 모름
4. **유지보수 지옥**: 신조어가 생길 때마다 사람이 사전을 계속 업데이트해야 함

> 💡 그래서 현대 NLP는 규칙을 사람이 쓰지 않습니다. **수백만 개의 문장을 모델이 직접 학습**해서 맥락을 파악하게 만듭니다. 오늘 마지막 유닛에서 그 모델을 직접 배치해보고, v1과 정면 대결을 붙여볼 겁니다. 🥊

## ⚠️ 자주 하는 실수

- **"감성 분석은 긍정/부정 2개뿐"** → 실무에서는 중립, 복합 감정(노래는 좋은데 안무는 별로), 감정 강도까지 다룹니다.
- **대소문자 처리 누락** → "Best"와 "best"는 컴퓨터에게 다른 문자열입니다. `.lower()`를 잊지 마세요.
- **`in` 연산의 함정** → `"art" in "started"`도 True입니다. 단어 단위 매칭이 필요한 이유이며, 다음 유닛(토큰화)에서 해결합니다.

## 🚀 실무에서는?

- **VOC(고객의 소리) 분석**: 통신사·카드사는 하루 수만 건의 상담 텍스트를 감성 분석해 불만 급증 상품을 실시간 탐지합니다.
- **엔터테인먼트 산업**: 실제로 기획사들은 컴백 직후 SNS 반응을 모니터링해 프로모션 방향을 조정합니다. 오늘 미션이 바로 그 축소판입니다.
- **앱 리뷰 자동 분류**: 부정 리뷰 중 '결제 오류' 키워드가 포함된 건만 개발팀에 자동 전달하는 파이프라인이 흔히 쓰입니다.

## 🧩 Check Point

**Q1. NLP의 핵심 과제를 가장 잘 표현한 것은?**

1. 글자를 더 빠르게 저장하는 것
2. 글자(기호)를 컴퓨터가 다룰 수 있는 **의미 있는 숫자**로 바꾸는 것
3. 모든 언어를 영어로 통일하는 것
4. 문법 오류를 자동으로 교정하는 것

**Q2. 규칙 기반 감성 분석(v1)이 실패하는 상황이 _아닌_ 것은?**

1. 부정어 결합 — "not bad"
2. 반어법 — "참 잘~한다"
3. 사전에 없는 신조어 — "개꿀"
4. 사전에 있는 긍정 단어가 그대로 쓰인 문장 — "this song is great"

**Q3. "Not bad at all"을 v1이 부정으로 판정한 이유는?**

1. 문장이 너무 짧아서
2. "bad"라는 단어만 보고 점수를 매겼을 뿐, **"not"과의 결합(맥락)을 보지 못해서**
3. 대문자 "Not"을 인식하지 못해서
4. 영어 사전이 없어서

<details>
<summary> ---------- ✅ 정답 확인 ---------- </summary>

**Q1 → 2번.** NLP의 출발점은 "글자 → 의미를 담은 숫자" 변환입니다. 오늘 하루가 전부 이 여정입니다(아스키 코드 → 토큰 → 임베딩).

**Q2 → 4번.** 사전에 있는 단어가 곧이곧대로 쓰인 문장은 v1이 잘 맞힙니다. 나머지는 전부 v1을 무너뜨린 함정들이죠.

**Q3 → 2번.** v1은 단어를 낱개로만 세기 때문에 "bad"에 −1점을 줄 뿐, 앞의 "not"이 의미를 뒤집는다는 것을 모릅니다. "맥락을 못 본다" — 오늘 오후 내내 이 문제를 해결하러 갑니다.

</details>

## 💻 실습 (자습용)

In [ ]:
# TODO 1 (사람의 판단): 아래 3개 문장을 여러분이 직접 긍정/부정/중립으로 판단해보세요 (주석으로 기록)
new_comments = [
    "콘서트 티켓팅 광탈했지만 노래는 인정이다",
    "This is fine I guess, nothing special though",
    "안무 영상 보고 바로 팬 됐어요 입덕 완료",
]



<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 예시 정답
# 1번: 긍정 (티켓팅 실패는 아쉽지만 핵심 메시지는 '노래 인정')
# 2번: 중립~약한 부정 ("fine I guess"는 미지근한 반응)
# 3번: 긍정 ('입덕'은 팬이 되었다는 강한 긍정 신호)
print("사람은 맥락으로 판단합니다. 이 직관을 기계에게 가르치는 것이 NLP입니다.")
```

</details>

In [ ]:
# new_comments에 있는 문장 3개를 판단

# 1번 문장:
# 2번 문장:
# 3번 문장:


In [ ]:
# TODO 2 (기계의 판단): sentiment_v1 함수로 위 3개 문장을 판정하고, 여러분의 판단과 비교해보세요


<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 예시 정답
for c in new_comments:
    print(f"[{sentiment_v1(c)}] {c}")
# '인정'이 사전에 있어 1번은 맞히지만, 2·3번은 사전에 단어가 없어 중립 처리됩니다.
```

</details>

In [ ]:
# 예시 정답





In [ ]:
# TODO 3 (한계 찾기): v1이 틀리게 판정할 것 같은 문장을 직접 2개 만들어 테스트해보세요
# 힌트: 부정어 뒤집기, 반어법, 신조어를 활용해보세요


<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 예시 정답
my_traps = [
    "최고라고는 못 하겠다",        # '최고'가 있지만 전체는 부정
    "이거 완전 띵곡이잖아",        # '띵곡'(명곡)은 사전에 없음 → 중립 처리
]
for c in my_traps:
    print(f"[{sentiment_v1(c)}] {c}")
```

</details>

In [ ]:
# 예시 정답





## 📝 실습 과제 (자습용)

In [ ]:
# 과제 1: positive_words / negative_words에 팬덤 신조어를 5개씩 추가해서
#         (예: 스밍, 입덕, 띵곡, 광탈, 탈덕 ...) v1의 정확도를 개선해보세요

# 과제 2: 좋아하는 콘텐츠(게임/드라마/음식 등)의 리뷰 문장을 5개 만들어
#         v1으로 판정해보고, 몇 개나 맞히는지 확인해보세요


<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 과제 1 예시 정답
positive_words += ["스밍", "입덕", "띵곡", "찢었다", "레전드"]
negative_words += ["탈덕", "노잼", "별점테러", "돈아깝", "실망각"]
print(f"[{sentiment_v1('이번 무대 완전 찢었다')}] 이번 무대 완전 찢었다")

# 과제 2 예시 정답
game_reviews = [
    "신규 업데이트 노잼이다 접는다",
    "이번 패치 레전드네 밸런스 완벽",
    "그래픽은 좋은데 스토리가 별로",
    "과금 유도 심해서 돈아깝다",
    "인생 게임 등극. 잠을 못 자겠어요",
]
for r in game_reviews:
    print(f"[{sentiment_v1(r)}] {r}")
```

</details>

In [ ]:
# 과제 1 예시 정답






In [ ]:
# 과제 2 예시 정답





## 📌 핵심 정리

- NLP = 사람의 언어를 컴퓨터가 다루게 하는 기술. 핵심 과제는 **글자 → 의미 있는 숫자** 변환
- 규칙 기반(단어 사전) 방식은 만들기 쉽지만 **부정어·반어법·신조어**에 취약하고 유지보수가 불가능에 가까움
- 현대 NLP는 규칙 대신 **데이터 학습**으로 맥락을 파악 — 오늘 마지막 유닛에서 직접 확인 예정
- 감성 분석기 v1은 오늘의 최종 대결(CH16-05)에 다시 등장합니다. 잊지 마세요 🥊

---
# 📘 CH16-02. 토큰화 — 문장을 AI의 조각으로

⏱️ **예상 소요시간**: 45분 (아이스브레이킹 10분 포함)

> 🎬 **미션 ②** — 팀장: "댓글을 통째로 다룰 순 없어요. AI가 소화할 수 있는 **조각(토큰)** 으로 잘라야 합니다. 그런데... 왜 같은 말도 한국어로 쓰면 AI 요금이 더 나오는지 아세요?"


## 🎮 아이스브레이킹 — Tiktokenizer로 토큰 눈으로 보기

🔗 **https://tiktokenizer.vercel.app** (GPT 계열 모델이 실제 사용하는 토크나이저를 브라우저에서 시각화)

**시연 시나리오**

1. 모델을 `gpt-4o` 로 선택하고 `I love NEBULA so much!` 입력 → 토큰이 색깔 블록으로 쪼개지는 것 관찰 (약 6~7토큰)
2. 같은 뜻의 한국어 `네뷸라 너무 사랑해요!` 입력 → **토큰 수가 훨씬 많아지는 것** 확인
3. 학생들에게 Zoom 채팅으로 "토큰 수가 가장 많이 나올 것 같은 짧은 문장"을 제안받아 즉석 테스트 (이모지, ㅋㅋㅋ, 신조어가 특히 재밌게 쪼개짐)
4. 훅 포인트: **"ChatGPT API는 토큰 수로 과금됩니다. 같은 질문도 한국어로 하면 더 비쌉니다. 왜 그런지 오늘 이 시간에 알게 됩니다."**

> 💡 사이트 접속이 안 될 경우: 아래 본문의 BERT 토크나이저 코드로 동일한 개념을 로컬에서 시연할 수 있습니다.

## 🤔 먼저 생각해보기

- "나는학교에간다"를 읽을 수 있나요? 띄어쓰기가 없어도 사람은 단어 경계를 찾아냅니다. 컴퓨터는 어떻게 찾을까요?
- 영어는 띄어쓰기로 자르면 될 것 같은데, "don't"는 한 단어일까요 두 단어일까요?
- 사전에 없는 새 단어("뉴진스러움", "Supernova-core")를 만나면 AI는 어떻게 처리할까요?

## 📖 핵심 개념

### 1) 토큰화(Tokenization) = 문장을 처리 단위로 자르기

AI는 문장을 통째로 이해하지 않습니다. **토큰(token)** 이라는 작은 조각으로 잘라서 처리합니다. 어떻게 자르느냐가 곧 모델의 성능과 비용을 좌우합니다.

가장 원시적인 방법부터:

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
comment = "The music video is absolutely stunning, best comeback ever"

# 방법 1: 공백 기준으로 자르기 (파이썬 기본 split)
tokens = comment.split()
print(tokens)
print(f"토큰 수: {len(tokens)}")
```

</details>

In [ ]:
comment = "The music video is absolutely stunning, best comeback ever"

# 방법 1: 공백 기준으로 자르기 (파이썬 기본 split)





<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 공백 분리의 문제: 문장부호가 단어에 붙어버림
messy = "Best comeback ever!!! Don't you agree?"
print(messy.split())
# 'ever!!!' 와 'ever'는 다른 토큰이 되어버림 → 같은 단어인데 따로 취급됨
```

</details>

In [ ]:
# 공백 분리의 문제: 문장부호가 단어에 붙어버림

messy = "Best comeback ever!!! Don't you agree?"


print(messy.split())

# 'ever!!!' 와 'ever'는 다른 토큰이 되어버림 → 같은 단어인데 따로 취급됨

### 2) 라이브러리 토큰화 — nltk

NLP 전용 라이브러리는 문장부호, 축약형(don't → do + n't)까지 규칙에 따라 정교하게 잘라줍니다.

NLTK(Natural Language Toolkit)는 자연어 처리를 배우기 위한 대표적인 파이썬 라이브러리입니다.

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
from nltk.tokenize import word_tokenize

print(word_tokenize(messy))
# ['Best', 'comeback', 'ever', '!', '!', '!', 'Do', "n't", 'you', 'agree', '?']
# 문장부호가 분리되고, Don't가 Do + n't로 나뉨
```

</details>

In [ ]:





# ['Best', 'comeback', 'ever', '!', '!', '!', 'Do', "n't", 'you', 'agree', '?']
# 문장부호가 분리되고, Don't가 Do + n't로 나뉨

### 3) 불용어(Stopwords) — 의미 분석에 방해되는 조연 걷어내기

"the", "is", "a" 같은 단어는 매우 자주 나오지만 감성·주제 판단에는 거의 기여하지 않습니다. 이런 단어를 **불용어**라 부르고, 분석 목적에 따라 제거합니다.

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
from nltk.corpus import stopwords

en_stop = set(stopwords.words("english"))
print(f"영어 불용어 개수: {len(en_stop)}")
print(list(en_stop)[:15])
```

</details>

In [ ]:
# 코드 입력





<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
comment = "The music video is absolutely stunning, best comeback ever"
tokens = word_tokenize(comment.lower())

filtered = [t for t in tokens if t not in en_stop and t.isalpha()]
print("원본 토큰:", tokens)
print("불용어 제거:", filtered)
# 핵심 의미 단어만 남음 → 어떤 댓글인지 한눈에 보임
```

</details>

In [ ]:
comment = "The music video is absolutely stunning, best comeback ever"

# 코드 입력



# isalpha() 알파벳 확인
# 핵심 의미 단어만 남음 → 어떤 댓글인지 한눈에 보임

### 4) 한국어는 왜 더 어려울까?

한국어는 **조사**가 단어에 딱 붙습니다. 공백으로 자르면 "노래가", "노래를", "노래는"이 전부 다른 토큰이 됩니다.

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
ko_comment = "노래가 좋아서 노래를 하루종일 들었다"

# 공백 분리: '노래가'와 '노래를'이 서로 다른 토큰으로 취급됨
print(ko_comment.split())

# 한국어는 형태소 분석기가 필요 (konlpy 등)
# from konlpy.tag import Okt
# okt = Okt()
# print(okt.morphs(ko_comment))  # ['노래', '가', '좋아서', '노래', '를', ...]
# → '노래'라는 같은 단어를 제대로 인식하게 됨
# (konlpy는 Java 설치가 필요해 오늘은 개념만 소개합니다)
```

</details>

In [ ]:
ko_comment = "노래가 좋아서 노래를 하루종일 들었다"

# 공백 분리: '노래가'와 '노래를'이 서로 다른 토큰으로 취급됨
print(ko_comment.split())

# 한국어는 형태소 분석기가 필요 (konlpy 등)
# from konlpy.tag import Okt
# okt = Okt()
# print(okt.morphs(ko_comment))  # ['노래', '가', '좋아서', '노래', '를', ...]
# → '노래'라는 같은 단어를 제대로 인식하게 됨
# (konlpy는 Java 설치가 필요해 오늘은 개념만 소개합니다)

### 5) 부분단어(Subword) 토큰화 — 최신 모델들의 해법

GPT, BERT 등 현대 모델은 단어보다 작은 **부분단어(subword)** 단위로 자릅니다.

핵심 아이디어: **자주 나오는 글자 조합은 한 덩어리로, 처음 보는 단어는 아는 조각들의 조합으로.**

이렇게 하면 사전에 없는 신조어도 "모르는 단어(UNK)"로 버리지 않고 처리할 수 있습니다.

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
from transformers import AutoTokenizer

bert_tok = AutoTokenizer.from_pretrained("bert-base-uncased")

# 흔한 단어는 통째로 1토큰
print(bert_tok.tokenize("love"))

# 긴 단어는 아는 조각들로 분해 (##은 '앞 토큰에 이어붙는 조각'이라는 표시)
print(bert_tok.tokenize("unbelievable"))
print(bert_tok.tokenize("internationalization"))
```

</details>

In [ ]:
# 라이브러리 불러오기




# 흔한 단어는 통째로 1토큰




# 긴 단어는 아는 조각들로 분해 (##은 '앞 토큰에 이어붙는 조각'이라는 표시)





<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 처음 보는 신조어도 조각으로 어떻게든 처리 가능!
made_up_words = ["Supernovacore", "nebulize", "stanning"]
for w in made_up_words:
    print(f"{w:20s} → {bert_tok.tokenize(w)}")
# 사전에 없는 단어인데도 UNK 없이 조각 조합으로 표현됨
```

</details>

In [ ]:
# 처음 보는 신조어도 조각으로 어떻게든 처리 가능!
made_up_words = ["Supernovacore", "nebulize", "stanning"]
for w in made_up_words:
    print(f"{w:20s} → {bert_tok.tokenize(w)}")

# 사전에 없는 단어인데도 UNK 없이 조각 조합으로 표현됨

### 6) 토큰 요금 계산기 — 아이스브레이킹의 답

아까 Tiktokenizer에서 본 현상을 코드로 재현해봅시다. **왜 한국어는 토큰이 더 많이 나올까요?**

BERT 토크나이저의 어휘 사전은 대부분 영어 데이터로 만들어졌습니다. 한국어 글자 조합은 사전에 통 단위로 등록된 것이 적어서, 훨씬 잘게 쪼개집니다.

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
pairs = [
    ("This song is a masterpiece!", "이 노래는 명곡이다!"),
    ("Streaming all day", "하루종일 스트리밍"),
    ("Best comeback ever", "역대 최고의 컴백"),
]

print(f"{'영어':35s} {'토큰':4s} | {'한국어':22s} {'토큰':4s}")
print("-" * 75)
total_en, total_ko = 0, 0
for en, ko in pairs:
    n_en, n_ko = len(bert_tok.tokenize(en)), len(bert_tok.tokenize(ko))
    total_en += n_en
    total_ko += n_ko
    print(f"{en:35s} {n_en:4d} | {ko:20s} {n_ko:4d}")

print("-" * 75)
print(f"합계: 영어 {total_en}토큰 vs 한국어 {total_ko}토큰")
print(f"→ 같은 의미인데 한국어가 약 {total_ko/total_en:.1f}배 더 많은 토큰 소모")
print("→ 토큰 단위로 과금하는 상용 AI API에서는 그만큼 요금 차이가 발생!")
```

</details>

In [ ]:
pairs = [
    ("This song is a masterpiece!", "이 노래는 명곡이다!"),
    ("Streaming all day", "하루종일 스트리밍"),
    ("Best comeback ever", "역대 최고의 컴백"),
]

print(f"{'영어':35s} {'토큰':4s} | {'한국어':22s} {'토큰':4s}")
print("-" * 75)

total_en, total_ko = 0, 0
for en, ko in pairs:
    n_en, n_ko = len(bert_tok.tokenize(en)), len(bert_tok.tokenize(ko))
    total_en += n_en
    total_ko += n_ko
    print(f"{en:35s} {n_en:4d} | {ko:20s} {n_ko:4d}")

print("-" * 75)
print(f"합계: 영어 {total_en}토큰 vs 한국어 {total_ko}토큰")
print(f"→ 같은 의미인데 한국어가 약 {total_ko/total_en:.1f}배 더 많은 토큰 소모")
print("→ 토큰 단위로 과금하는 상용 AI API에서는 그만큼 요금 차이가 발생!")

> 💡 실제 상용 모델들은 다국어 데이터를 반영한 더 큰 어휘 사전을 사용해 격차가 줄고 있지만, 여전히 한국어가 영어보다 토큰 효율이 낮은 편입니다. 프롬프트를 영어로 쓰면 비용이 절감되는 경우가 있는 이유입니다.

## ⚠️ 자주 하는 실수

- **토큰 = 단어라고 착각** → 최신 모델의 토큰은 단어보다 작은 조각(subword)입니다. "internationalization"은 1단어지만 여러 토큰입니다.
- **한국어를 공백으로만 자르기** → 조사 때문에 같은 단어가 수십 가지 변형으로 흩어집니다.
- **불용어를 무조건 제거** → "not"은 불용어 목록에 있지만, 감성 분석에서 제거하면 "not bad"의 의미가 뒤집힙니다. **분석 목적에 따라 판단**해야 합니다.

## 🚀 실무에서는?

- **API 비용 최적화**: 토큰 수 = 돈. 기업들은 프롬프트를 다듬어 토큰을 줄이는 것만으로 월 수백만 원을 절감하기도 합니다.
- **Context 한도 관리**: 모델이 한 번에 처리 가능한 길이도 토큰 단위로 정해집니다. 긴 문서 처리 설계의 출발점이 토큰 계산입니다.
- **검색 품질**: 검색엔진·챗봇의 색인 품질은 토큰화 품질에서 시작됩니다. 한국어 서비스가 형태소 분석기에 투자하는 이유입니다.

## 🧩 Check Point

**Q1. 공백 분리 토큰화의 한계로 옳은 것을 _모두_ 고르면?**

1. 문장부호가 단어에 붙어버린다 ("stunning," ≠ "stunning")
2. 토큰 수가 항상 0이 된다
3. 한국어처럼 조사가 붙는 언어에서 같은 단어가 다른 토큰이 된다 ("노래가" ≠ "노래를")
4. 대문자를 처리할 수 없다

**Q2. subword 토큰화가 신조어에 강한 이유는?**

1. 신조어 사전을 매일 업데이트하기 때문에
2. 모르는 단어는 무조건 삭제하기 때문에
3. 사전에 없는 단어라도 **아는 조각(부분단어)들로 쪼개서** 어떻게든 표현할 수 있기 때문에
4. 신조어를 모두 영어로 번역하기 때문에

**Q3. 같은 의미인데 한국어가 영어보다 토큰이 많이 나오는 (전통적인) 이유는?**

1. 한국어가 영어보다 문법적으로 열등해서
2. 토크나이저의 어휘 사전이 **영어 중심 데이터**로 만들어져, 한국어는 잘게 쪼개질 수밖에 없어서
3. 한글은 컴퓨터가 저장할 수 없는 문자라서
4. 한국어 문장이 항상 물리적으로 더 길어서

<details>
<summary> ---------- ✅ 정답 확인 ---------- </summary>

**Q1 → 1, 3번.** 둘 다 수업에서 직접 확인한 한계입니다. (2, 4번은 사실이 아닙니다.)

**Q2 → 3번.** "un + happi + ness"처럼 조각으로 분해하는 능력 덕분에 어휘 사전 밖(OOV) 문제가 사라집니다.

**Q3 → 2번.** 언어의 우열이 아니라 **어휘 사전 구성의 문제**입니다. 그래서 최신 모델들(GPT-4o 등)이 다국어 데이터로 어휘 사전을 키우면서 이 격차는 실제로 줄어들고 있습니다 — 아이스브레이킹에서 확인한 그대로입니다.

</details>

## 💻 실습 (자습용)

In [ ]:
# TODO 1 (기본 토큰화): "NEBULA's new song is absolutely amazing!!!" 를
#        (a) split()  (b) word_tokenize 로 각각 토큰화하고 결과를 비교하세요


<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 예시 정답
s = "NEBULA's new song is absolutely amazing!!!"
print("split():      ", s.split())
print("word_tokenize:", word_tokenize(s))
# split은 amazing!!!이 한 덩어리, word_tokenize는 문장부호와 's를 분리
```

</details>

In [ ]:
# 코드 입력





# split은 amazing!!!이 한 덩어리, word_tokenize는 문장부호와 's를 분리

In [ ]:
# TODO 2 (불용어 제거): 위 문장을 소문자화 → 토큰화 → 영어 불용어 제거까지 수행하세요


<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 예시 정답
tokens = word_tokenize(s.lower())
filtered = [t for t in tokens if t not in en_stop and t.isalpha()]
print(filtered)
```

</details>

In [ ]:
# 코드 입력





In [ ]:
# TODO 3 (subword): "unhappiness", "stanning", "daebak" 세 단어를
#        bert-base-uncased 토크나이저로 토큰화하고 각각 몇 조각으로 나뉘는지 확인하세요


<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 예시 정답
for w in ["unhappiness", "stanning", "daebak"]:
    toks = bert_tok.tokenize(w)
    print(f"{w:12s} → {len(toks)}조각 {toks}")
```

</details>

In [ ]:
# 코드 입력





## 📝 실습 과제 (자습용)

In [ ]:
# 과제 1: 좋아하는 노래 제목이나 문장 5개를 골라 영어/한국어 버전의 토큰 수를 비교하는
#         나만의 '토큰 요금표'를 만들어보세요

# 과제 2: 불용어 제거가 오히려 의미를 훼손하는 문장을 2개 만들어보고,
#         어떤 불용어가 문제였는지 분석해보세요 (힌트: not, no, never)


<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 과제 1 예시 정답
my_pairs = [("I miss you", "보고 싶다"), ("Good night", "잘 자요")]
for en, ko in my_pairs:
    print(f"{en} [{len(bert_tok.tokenize(en))}] vs {ko} [{len(bert_tok.tokenize(ko))}]")

# 과제 2 예시 정답
s2 = "this is not good at all"
toks2 = [t for t in word_tokenize(s2) if t not in en_stop]
print(toks2)  # ['good'] 만 남음 → 부정문이 긍정처럼 보이게 됨! 'not'이 불용어였기 때문
```

</details>

In [ ]:
# 과제 1 정답





In [ ]:
# 과제 2 정답





## 📌 핵심 정리

- 토큰화 = 문장을 AI의 처리 단위로 자르는 작업. **어떻게 자르느냐가 성능과 비용을 결정**
- 공백 분리 → nltk 규칙 토큰화 → **subword(부분단어)** 순으로 진화. 최신 모델은 전부 subword 방식
- subword는 신조어도 아는 조각의 조합으로 처리 → UNK 문제 해결
- 한국어는 영어 중심 어휘 사전에서 잘게 쪼개져 **같은 뜻이라도 토큰 수(=API 요금)가 더 많음**
- 불용어 제거는 목적에 따라: 감성 분석에서 'not' 제거는 치명적

---
# 📘 CH16-03. 임베딩 — 단어에게 좌표를 주다

⏱️ **예상 소요시간**: 50분

> 🎬 **미션 ③** — 팀장: "'띵곡이다'와 '명곡이네요'는 같은 뜻인데, 컴퓨터는 글자가 달라서 남남으로 취급해요. **비슷한 의미의 댓글끼리 자동으로 묶으려면** 단어에 '의미의 좌표'를 부여해야 합니다."


## 🎮 아이스브레이킹 — Semantris: AI와 단어 연상 게임

🔗 **https://research.google.com/semantris/** (Google AI의 단어 연상 게임 — 임베딩 유사도로 작동)

**시연 시나리오**

1. **Blocks 모드** 선택 (턴제라 시연에 적합). 화면의 단어 블록 중 하나를 골라, 그 단어를 직접 치지 않고 **연상되는 단어**를 입력
2. 예: 화면에 `coffee`가 있으면 → `morning`이라고 입력 → AI가 coffee 블록을 알아서 지목하는 것 관찰
3. 학생들에게 Zoom 채팅으로 힌트 단어를 제안받아 2~3턴 진행 (엉뚱한 연상어로 AI를 시험해보게 유도)
4. 훅 포인트: **"AI는 'coffee'와 'morning'이 가깝다는 걸 어떻게 알까요? 두 단어 사이의 '거리'를 계산했기 때문입니다. 단어에 좌표가 있다는 뜻이죠. 오늘 그 좌표를 직접 만져봅니다."**

> 💡 접속 불가 시: 아래 본문의 GloVe `most_similar` 코드로 동일한 '연상 게임'을 로컬에서 재현할 수 있습니다.

## 🤔 먼저 생각해보기

- "왕 - 남자 + 여자 = ?" 라는 연산이 가능하다면 답은 뭘까요? 단어끼리 덧셈·뺄셈이 된다는 게 말이 될까요?
- 지도에서 서울과 인천은 가깝고 서울과 부산은 멉니다. 단어의 세계에도 이런 '지도'를 만들 수 있을까요?
- '띵곡'과 '명곡'이 비슷하다는 것을, 사전 없이 데이터만으로 알아낼 방법이 있을까요?

## 📖 핵심 개념

### 1) One-Hot 인코딩의 치명적 문제

단어를 숫자로 바꾸는 가장 단순한 방법: 단어마다 자리 하나씩 배정하고, 해당 자리만 1로 켜기(One-Hot).

그런데 이 방식엔 치명적 문제가 있습니다. 직접 확인해봅시다.

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
import numpy as np

# 어휘 사전: [king, queen, apple] 3단어 → 각 단어는 3차원 One-Hot 벡터
king  = np.array([1, 0, 0])
queen = np.array([0, 1, 0])
apple = np.array([0, 0, 1])

# 단어 사이의 거리 계산
print("king ↔ queen 거리:", np.linalg.norm(king - queen))
print("king ↔ apple 거리:", np.linalg.norm(king - apple))
# 완전히 동일! One-Hot은 '왕과 여왕이 왕과 사과보다 가깝다'는 사실을 전혀 표현하지 못함
```

</details>

In [ ]:
import numpy as np

# 어휘 사전: [king, queen, apple] 3단어 → 각 단어는 3차원 One-Hot 벡터
king  = np.array([1, 0, 0])
queen = np.array([0, 1, 0])
apple = np.array([0, 0, 1])

# 단어 사이의 거리 계산
# linear Algebra 선형대수 (벡터 및 행렬 연산)
# norm 벡터 길이, 사이 거리 계산

print("king ↔ queen 거리:", np.linalg.norm(king - queen))
print("king ↔ apple 거리:", np.linalg.norm(king - apple))
# 완전히 동일! One-Hot은 '왕과 여왕이 왕과 사과보다 가깝다'는 사실을 전혀 표현하지 못함

모든 단어 쌍의 거리가 똑같다 = **의미 관계가 0% 반영된 표현**입니다. 게다가 단어가 10만 개면 벡터도 10만 차원이 됩니다.

### 2) 임베딩(Embedding) — 의미를 좌표에 담다

**임베딩** = 단어를 낮은 차원(보통 50~1024차원)의 **실수 벡터**로 표현하되, **의미가 비슷한 단어는 가까운 좌표**에 배치하는 기법.

핵심 원리는 단순하고 아름답습니다:

> **"비슷한 문맥에 등장하는 단어는 비슷한 의미다."**
> ("나는 오늘 ___를 마셨다" 빈칸에 들어가는 단어들 — 커피, 차, 물 — 은 서로 비슷한 의미)

수백만 문장에서 이 통계를 학습하면, 사람이 사전을 안 만들어도 단어 지도가 저절로 그려집니다.

### 3) 진짜 임베딩 만져보기 — GloVe

GloVe는 위키백과 등 방대한 텍스트로 미리 학습된 공개 단어 임베딩입니다. 50차원 버전을 로딩합니다.

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
import gensim.downloader as api

# 사전 다운로드 셀을 실행했다면 캐시에서 즉시 로딩됩니다
glove = api.load("glove-wiki-gigaword-50")
print(f"어휘 수: {len(glove.key_to_index):,}개")

# 단어 하나는 50개의 실수 = 50차원 공간의 좌표
print("\n'music'의 좌표 (앞 10개 차원만):")
print(glove["music"][:10])
```

</details>

In [ ]:
import gensim.downloader as api
# 1~2분 실행

# 사전 다운로드 셀을 실행했다면 캐시에서 즉시 로딩됩니다
#glove = api.load("glove-wiki-gigaword-50")

In [ ]:
# 이 모델에 저장된 단어(어휘)가 몇개인지 출력




# 단어 하나는 50개의 실수 = 50차원 공간의 좌표






<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 이 좌표가 정말 '의미'를 담고 있을까? 가까운 이웃을 조회해보자
print("music과 가까운 단어들:")
for word, sim in glove.most_similar("music", topn=5):
    print(f"  {word:12s} 유사도 {sim:.3f}")

print("\nconcert와 가까운 단어들:")
for word, sim in glove.most_similar("concert", topn=5):
    print(f"  {word:12s} 유사도 {sim:.3f}")
```

</details>

In [ ]:
# 이 좌표가 정말 '의미'를 담고 있을까? 가까운 이웃을 조회해보자







### 4) 단어 연금술 — 벡터 연산으로 의미 조작하기

임베딩의 가장 유명한 마법: **좌표이기 때문에 덧셈·뺄셈이 가능**합니다.

`king - man + woman ≈ ?` — "왕에서 남성성을 빼고 여성성을 더하면?"이라는 의미 연산이 실제로 계산됩니다.

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
def word_math(positive, negative):
    result = glove.most_similar(positive=positive, negative=negative, topn=3)
    expr = " + ".join(positive) + " - " + " - ".join(negative)
    print(f"{expr} = ?")
    for word, sim in result:
        print(f"  → {word} ({sim:.3f})")
    print()

word_math(["king", "woman"], ["man"])       # 왕 - 남자 + 여자
word_math(["paris", "korea"], ["france"])   # 파리 - 프랑스 + 한국
word_math(["singer", "dance"], ["song"])    # 가수 - 노래 + 춤
```

</details>

In [ ]:
def word_math(positive, negative):
    result = glove.most_similar(positive=positive, negative=negative, topn=3)
    expr = " + ".join(positive) + " - " + " - ".join(negative)
    print(f"{expr} = ?")
    for word, sim in result:
        print(f"  → {word} ({sim:.3f})")
    print()

word_math(["king", "woman"], ["man"])       # 왕 - 남자 + 여자
word_math(["paris", "korea"], ["france"])   # 파리 - 프랑스 + 한국
word_math(["singer", "dance"], ["song"])    # 가수 - 노래 + 춤

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 🧪 단어 연금술 자유 실험 — 재밌는 조합을 직접 시도해보세요!
# (Zoom 채팅으로 학생들에게 조합을 제안받아 즉석 실험해보기 좋은 셀)

word_math(["pizza", "korea"], ["italy"])
word_math(["coffee", "night"], ["morning"])
```

</details>

In [ ]:
# 🧪 단어 연금술 자유 실험 — 재밌는 조합을 직접 시도해보세요!






### 5) 단어 지도 그리기 — 50차원을 2차원으로

50차원 좌표는 눈으로 볼 수 없으니, PCA로 2차원에 눌러 담아 **단어 지도**를 그려봅시다. 의미가 비슷한 단어끼리 실제로 뭉치는지 확인!

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# 세 그룹의 단어: 음악 / 음식 / 감정
words = ["music", "song", "concert", "album", "singer",
         "pizza", "coffee", "burger", "noodle", "cake",
         "happy", "sad", "angry", "excited", "lonely"]

vectors = np.array([glove[w] for w in words])
coords = PCA(n_components=2).fit_transform(vectors)

plt.figure(figsize=(9, 6))
colors = ["#2E4BE8"]*5 + ["#E8930C"]*5 + ["#111C2E"]*5
plt.scatter(coords[:, 0], coords[:, 1], c=colors, s=80)
for i, w in enumerate(words):
    plt.annotate(w, (coords[i, 0]+0.05, coords[i, 1]+0.05), fontsize=11)
plt.title("Word Map: music / food / emotion clusters")
plt.grid(alpha=0.3)
plt.show()
# 음악 단어끼리, 음식 단어끼리, 감정 단어끼리 뭉쳐 있는 것을 확인!
```

</details>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# 세 그룹의 단어: 음악 / 음식 / 감정
words = ["music", "song", "concert", "album", "singer",
         "pizza", "coffee", "burger", "noodle", "cake",
         "happy", "sad", "angry", "excited", "lonely"]

# 각 단어의 50차원 임베딩 벡터를 가져와 하나의 배열로 만듦
# 결과: (15개 단어 × 50차원) 형태의 행렬




# 위에서 가지고온 vetors를 2차원으로 축소






plt.figure(figsize=(9, 6))
# 그룹별 색상 지정
colors = ["#2E4BE8"]*5 + ["#E8930C"]*5 + ["#111C2E"]*5

# 위에서 저장한 coords의 첫번째 열 = x축, 두번 째 열 y 축 좌표로 사용
plt.scatter(coords[:, 0], coords[:, 1], c=colors, s=80)

# 점들들 좌표에 겹치지 않게 표시
for i, w in enumerate(words):
    plt.annotate(w, (coords[i, 0]+0.05, coords[i, 1]+0.05), fontsize=11)
plt.title("Word Map: music / food / emotion clusters")
plt.grid(alpha=0.3)
plt.show()
# 음악 단어끼리, 음식 단어끼리, 감정 단어끼리 뭉쳐 있는 것을 확인!

### 6) 🎮 보조 시연 — Embedding Projector

🔗 **https://projector.tensorflow.org** → 좌측 데이터를 `Word2Vec 10K`로 선택

- 1만 개 단어가 3D 공간에 떠 있는 모습을 회전하며 탐색
- 우측 검색창에 `music` 입력 → 가까운 이웃 단어들이 하이라이트되는 것 관찰
- 방금 우리가 matplotlib로 그린 단어 지도의 **초대형 실사판**입니다

### 7) 그래서 팬 댓글 분류는?

오늘 배운 것은 **단어** 임베딩입니다. "띵곡이다"라는 **문장 전체**를 하나의 좌표로 만드는 **문장 임베딩**으로 확장하면, 비슷한 의미의 댓글끼리 자동으로 묶을 수 있습니다. 문장 임베딩은 이후 RAG 수업에서 주인공으로 다시 등장합니다.

## ⚠️ 자주 하는 실수

- **임베딩 차원 수를 어휘 수와 혼동** → One-Hot은 어휘 수 = 차원 수지만, 임베딩은 어휘가 40만 개여도 50차원으로 표현 가능합니다.
- **유사도와 거리를 혼동** → 코사인 유사도는 클수록 비슷, 거리는 작을수록 비슷. 반대 방향입니다.
- **임베딩이 항상 옳다고 믿기** → 학습 데이터의 편향이 그대로 좌표에 새겨집니다(예: 직업-성별 편향). 실무에서 중요한 검증 포인트입니다.

## 🚀 실무에서는?

- **추천 시스템**: "이 상품을 본 사람이 함께 본 상품"의 상당수가 임베딩 유사도 기반입니다.
- **검색 고도화**: '노트북 가방'을 검색하면 '랩탑 파우치'도 나오는 것 — 키워드가 아닌 임베딩 매칭 덕분입니다.
- **이상 탐지**: 고객 문의 임베딩이 기존 클러스터와 동떨어진 위치에 찍히면 → 신규 유형의 이슈 발생 신호로 활용합니다.

## 🧩 Check Point

**Q1. One-Hot 인코딩이 의미 표현에 실패하는 이유는?**

1. 벡터가 너무 짧아서
2. 숫자가 아닌 문자로 저장되어서
3. 모든 단어 쌍의 거리가 똑같아서 — **의미의 가깝고 멂이 전혀 반영되지 않아서**
4. 긍정 단어만 표현할 수 있어서

**Q2. 임베딩 학습의 핵심 가정(분포 가설)을 가장 잘 표현한 것은?**

1. 자주 등장하는 단어일수록 중요하다
2. **비슷한 문맥에서 쓰이는 단어는 비슷한 의미를 가진다**
3. 긴 단어일수록 복잡한 의미를 가진다
4. 단어의 의미는 사전에 정의된 대로 고정된다

**Q3. king − man + woman ≈ queen 연산이 가능한 이유는?**

1. 컴퓨터에 왕실 족보 데이터가 들어있어서
2. 우연의 일치일 뿐, 다른 단어로는 재현되지 않아서
3. 알파벳 순서가 비슷해서
4. 임베딩 공간에서 **의미 관계(성별, 신분 등)가 일정한 '방향'으로 학습되어** 벡터 덧셈·뺄셈으로 조작할 수 있어서

<details>
<summary> ---------- ✅ 정답 확인 ---------- </summary>

**Q1 → 3번.** 모든 단어가 서로 직각(거리 동일)이니 "cat과 dog이 가깝다"는 정보가 0%입니다. 차원 폭발은 덤이고요.

**Q2 → 2번.** "단어의 의미는 그 단어가 어울리는 친구들을 보면 안다" — GloVe도, GPT의 임베딩도 이 한 문장 위에 서 있습니다.

**Q3 → 4번.** '남→여' 방향 벡터가 공간 전체에서 대체로 일정하게 학습되기 때문입니다. tokyo − japan + france ≈ paris 로도 재현되는, 우연이 아닌 구조입니다.

</details>

## 💻 실습 (자습용)

In [ ]:
# TODO 1 (이웃 탐색): 'summer'와 가장 가까운 단어 5개를 조회해보세요


<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 예시 정답
for word, sim in glove.most_similar("summer", topn=5):
    print(f"{word:12s} {sim:.3f}")
```

</details>

In [ ]:
# 코드 입력




In [ ]:
# TODO 2 (유사도 비교): 'dog'과 'cat'의 유사도 vs 'dog'과 'pizza'의 유사도를 비교하세요
#        (힌트: glove.similarity(단어1, 단어2))


<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 예시 정답
print("dog ↔ cat  :", glove.similarity("dog", "cat"))
print("dog ↔ pizza:", glove.similarity("dog", "pizza"))
```

</details>

In [ ]:
# 코드 입력




In [ ]:
# TODO 3 (단어 연금술): 'tokyo' - 'japan' + 'france' 연산의 결과를 확인해보세요


<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 예시 정답
word_math(["tokyo", "france"], ["japan"])   # → paris 계열이 상위에 등장
```

</details>

In [ ]:
# 코드 입력




## 📝 실습 과제 (자습용)

In [ ]:
# 과제 1: 좋아하는 분야(게임/스포츠/영화 등)의 영어 단어 10개로
#         나만의 단어 지도를 PCA 2차원 시각화로 그려보세요

# 과제 2: 단어 연금술 조합을 5개 직접 설계해 실험하고,
#         가장 그럴듯한 결과와 가장 엉뚱한 결과를 각각 기록해보세요


<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 과제 1 예시 정답
game_words = ["game", "player", "level", "boss", "quest",
              "soccer", "goal", "team", "coach", "stadium"]
vv = np.array([glove[w] for w in game_words])
cc = PCA(n_components=2).fit_transform(vv)
plt.figure(figsize=(8, 5))
plt.scatter(cc[:, 0], cc[:, 1], c=["#2E4BE8"]*5 + ["#E8930C"]*5, s=80)
for i, w in enumerate(game_words):
    plt.annotate(w, (cc[i, 0]+0.05, cc[i, 1]+0.05))
plt.grid(alpha=0.3); plt.show()

# 과제 2 예시 정답
word_math(["actor", "music"], ["movie"])
word_math(["breakfast", "night"], ["morning"])
```

</details>

In [ ]:
# 과제 1 예시 정답








In [ ]:
# 과제 2 예시 정답





## 📌 핵심 정리

- One-Hot은 모든 단어 쌍의 거리가 동일 → **의미 관계 표현 불가**
- 임베딩 = 의미가 비슷한 단어를 가까운 좌표에 배치한 **저차원 실수 벡터**. "비슷한 문맥 = 비슷한 의미"라는 분포 가설로 학습
- 좌표이기 때문에 **연산이 가능**: king - man + woman ≈ queen
- Semantris, 추천 시스템, 의미 검색 — 모두 임베딩 공간에서의 **거리 계산**이 본질
- 문장 단위 임베딩은 이후 RAG 수업에서 본격 활용

---
# 📘 CH16-04. Transformer와 GPT의 속마음

⏱️ **예상 소요시간**: 45분 (아이스브레이킹 10분 포함)

> 🎬 **미션 ④** — 팀장: "해외 팬 댓글 번역이랑 답글 자동 생성 기능을 검토 중이에요. 요즘 번역기랑 ChatGPT는 대체 **속에서 무슨 일이 벌어지길래** 이렇게 잘하는 거죠? 엔진을 열어봅시다."


## 🎮 아이스브레이킹 — GPT의 내부를 3D로 걸어서 구경하기

🔗 **https://bbycroft.net/llm** (LLM Visualization — GPT 내부 구조를 3D 건축물처럼 탐험)

**시연 시나리오**

1. 첫 화면의 가장 작은 모델(nano-GPT)에서 시작 — 글자가 입력되어 예측이 나오기까지의 전체 흐름을 카메라로 따라가며 관람
2. 스크롤을 내리면 Embedding → Attention(Self-Attention) → 출력층 순서로 각 부품이 확대됨.
3. 마지막에 우측의 **GPT-3 모델과 크기 비교**
4. 훅 포인트: **"ChatGPT도 구조는 이 작은 모델과 똑같습니다. 크기만 수십만 배 커졌을 뿐. 오늘 이 구조의 핵심 부품인 Attention을 이해하면, 최신 AI 뉴스가 전부 읽히기 시작합니다."**

> 💡 백업 사이트: **https://poloclub.github.io/transformer-explainer/** — 브라우저 안에서 GPT-2가 실제로 돌아가며 attention을 시각화해주는 인터랙티브 데모 (직접 문장 입력 가능)

## 🤔 먼저 생각해보기

- "나는 배가 고파서 배를 타고 배를 사러 갔다" — 사람은 세 개의 '배'를 어떻게 구분할까요?
- 책을 한 글자씩 순서대로만 읽어야 한다면 vs 페이지 전체를 한눈에 보며 읽는다면, 어느 쪽이 빠르고 정확할까요?
- ChatGPT는 답을 통째로 만들어낼까요, 아니면 한 단어씩 만들까요?

## 📖 핵심 개념

### 1) Transformer 이전 — 순서대로 읽기의 한계

Transformer 이전의 모델(RNN 계열)은 문장을 **한 단어씩 순서대로** 읽었습니다.

- 🐌 **느리다**: 100번째 단어를 처리하려면 앞의 99개를 다 거쳐야 함 (병렬 처리 불가)
- 🧠 **잊는다**: 문장이 길어지면 앞부분의 정보가 희미해짐 ("장기 기억" 문제)

2017년, 구글의 논문 *Attention Is All You Need*가 판을 뒤집습니다. **"순서대로 읽지 말고, 문장 전체를 한 번에 보면서 단어들끼리 서로 참조하게 하자"** — 이것이 **Transformer**입니다.

### 2) Attention — 단어들이 서로에게 보내는 시선

**Attention(어텐션)** = 각 단어가 문장 안의 다른 단어들을 얼마나 참고할지 **가중치**로 계산하는 장치.

예: "The concert was not boring at all"에서 `boring`의 의미를 해석하려면 `not`을 강하게 참조해야 합니다. Attention이 바로 그 시선의 세기를 학습합니다.

말로만 하지 말고, 실제 모델의 attention을 열어봅시다.

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
import torch
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModel

# 사전학습된 모델을 attention 출력 모드로 로딩
model_name = "distilbert-base-uncased-finetuned-sst-2-english"
tok = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name, output_attentions=True)

sentence = "the concert was not boring at all"
inputs = tok(sentence, return_tensors="pt")
with torch.no_grad():
    outputs = model(**inputs)

tokens = tok.convert_ids_to_tokens(inputs["input_ids"][0])
print("토큰:", tokens)
print("attention 층 수:", len(outputs.attentions))
```

</details>

In [ ]:
import torch
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModel

# 사전학습된 모델을 attention 출력 모드로 로딩
model_name = "distilbert-base-uncased-finetuned-sst-2-english"
tok = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name, output_attentions=True)

sentence = "the concert was not boring at all"
inputs = tok(sentence, return_tensors="pt")
with torch.no_grad():
    outputs = model(**inputs)

tokens = tok.convert_ids_to_tokens(inputs["input_ids"][0])
print("토큰:", tokens)
print("attention 층 수:", len(outputs.attentions))

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 마지막 층의 attention을 heatmap으로 시각화 (여러 head의 평균)
att = outputs.attentions[-1][0].mean(dim=0).numpy()

plt.figure(figsize=(7, 6))
plt.imshow(att, cmap="Blues")
plt.xticks(range(len(tokens)), tokens, rotation=45)
plt.yticks(range(len(tokens)), tokens)
plt.title("Attention Map: which word looks at which?")
plt.colorbar(label="attention weight")
plt.tight_layout()
plt.show()
# 세로축의 단어가 가로축의 단어를 얼마나 '쳐다보는지'의 세기
# 'boring' 행에서 'not' 방향의 값이 밝게 나타나는지 관찰해보세요
```

</details>

In [ ]:
# 마지막 층의 attention을 heatmap으로 시각화 (여러 head의 평균)
att = outputs.attentions[-1][0].mean(dim=0).numpy()

plt.figure(figsize=(7, 6))

plt.imshow(att, cmap="Blues")

plt.xticks(range(len(tokens)), tokens, rotation=45)
plt.yticks(range(len(tokens)), tokens)

plt.title("Attention Map: which word looks at which?")
plt.colorbar(label="attention weight")

plt.tight_layout()
plt.show()
# 세로축의 단어가 가로축의 단어를 얼마나 '쳐다보는지'의 세기
# 'boring' 행에서 'not' 방향의 값이 밝게 나타나는지 관찰해보세요

### 3) Transformer 구조 한 장 정리

```
입력 문장
   ↓
[토큰화]           ← CH16-02에서 배운 그것
   ↓
[임베딩]           ← CH16-03에서 배운 그것 (+ 위치 정보 추가)
   ↓
[Attention 블록] × N층   ← 단어들이 서로 참조하며 문맥 정보를 섞음
   ↓
[출력층]           ← 태스크에 맞는 결과 (분류, 다음 단어 예측 등)
```

오늘 오전 내내 배운 것들이 그대로 Transformer의 앞단 부품이었습니다.

- **병렬 처리 가능** → GPU로 학습 속도 폭발 → 모델 크기 폭발 → 지금의 LLM 시대
- BERT(문장 이해 특화), GPT(생성 특화) 등 현대 언어 모델은 전부 Transformer의 변형입니다.

### 4) GPT의 정체 — "다음 단어 맞히기" 기계

GPT(Generative Pre-trained Transformer)가 하는 일은 놀랍도록 단순합니다:

> **"지금까지의 단어들을 보고, 다음 단어 하나를 확률로 예측한다. 그리고 그걸 반복한다."**

정말인지 GPT-2의 머릿속 확률을 직접 들여다봅시다.
- (구조가 단순)
- 모델 크기가 작음 (약 1억 2,400만개의 파라미터)
- 최신 모델은 수십억~수천억개의 파라미터라서 교육용으로는 X

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
from transformers import GPT2LMHeadModel, GPT2Tokenizer

gpt2_tok = GPT2Tokenizer.from_pretrained("gpt2")
gpt2 = GPT2LMHeadModel.from_pretrained("gpt2")

prompt = "The best thing about music is"
inputs = gpt2_tok(prompt, return_tensors="pt")
with torch.no_grad():
    logits = gpt2(**inputs).logits

# 마지막 위치에서 '다음 단어' 확률 상위 5개
probs = torch.softmax(logits[0, -1], dim=-1)
top = torch.topk(probs, 5)

print(f"프롬프트: '{prompt}' 다음에 올 단어 후보 TOP 5")
for p, idx in zip(top.values, top.indices):
    print(f"  '{gpt2_tok.decode(idx).strip()}'  확률 {p.item()*100:.1f}%")
```

</details>

In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

gpt2_tok = GPT2Tokenizer.from_pretrained("gpt2")
gpt2 = GPT2LMHeadModel.from_pretrained("gpt2")

In [ ]:
# 코드 입력




# 마지막 위치에서 '다음 단어' 확률 상위 5개





print(f"프롬프트: '{prompt}' 다음에 올 단어 후보 TOP 5")
for p, idx in zip(top.values, top.indices):
    print(f"  '{gpt2_tok.decode(idx).strip()}'  확률 {p.item()*100:.1f}%")

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 예측을 반복하면 = 문장 생성!
output = gpt2.generate(
    **inputs,
    max_new_tokens=25,
    do_sample=True,       # 확률적으로 샘플링 (창의적 생성)
    temperature=0.8,
    pad_token_id=gpt2_tok.eos_token_id,
)
print(gpt2_tok.decode(output[0]))
```

</details>

In [ ]:
# 예측을 반복하면 = 문장 생성!
# 참고: GPT2는 2019년 모델로 한계점이 명확 (문장이 다소 어색함)






In [ ]:
prompt = "The capital of France is"

inputs = gpt2_tok(prompt, return_tensors="pt")
with torch.no_grad():
    logits = gpt2(**inputs).logits

# 마지막 위치에서 '다음 단어' 확률 상위 5개
probs = torch.softmax(logits[0, -1], dim=-1)
top = torch.topk(probs, 5)

print(f"프롬프트: '{prompt}' 다음에 올 단어 후보 TOP 5")
for p, idx in zip(top.values, top.indices):
    print(f"  '{gpt2_tok.decode(idx).strip()}'  확률 {p.item()*100:.1f}%")

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 같은 프롬프트, 다른 temperature — 창의성 다이얼 돌려보기
for temp in [0.3, 1.2]:
    output = gpt2.generate(**inputs, max_new_tokens=20, do_sample=True,
                           temperature=temp, pad_token_id=gpt2_tok.eos_token_id)
    print(f"[temperature={temp}]")
    print(" ", gpt2_tok.decode(output[0]))
    print()
# 낮은 temperature: 무난하고 예측 가능 / 높은 temperature: 다양하지만 산으로 갈 위험
```

</details>

In [ ]:
# 같은 프롬프트, 다른 temperature — 창의성 다이얼 돌려보기
# Temperature는 확률 분포를 얼마나 '평평하게' 만들지 결정하는 '창의성' 옵션
# Temperature = 0.3 (낮음) > 확률이 높은 단어를 거의 항상 선택
# Temperature = 1.2 (높음) > 확률이 낮은 단어도 선택





# 낮은 temperature: 무난하고 예측 가능 / 높은 temperature: 다양하지만 산으로 갈 위험


### 5) "생각보다 별로인데요?" — 크기가 곧 실력

GPT-2의 출력을 보고 실망했을 수 있습니다. 당연합니다. **원리는 최신 모델과 동일하지만 크기가 다릅니다.**

| 모델 | 파라미터 수 | 비고 |
|---|---|---|
| GPT-2 (오늘 실습) | 1.24억 | 노트북에서 무료로 도는 교육용 크기 |
| GPT-2 XL | 15억 | |
| GPT-3 | 1,750억 | ChatGPT 초기 버전의 뿌리 |
| 최신 상용 모델들 | 수천억~ (비공개) | 아이스브레이킹에서 본 그 마천루 |

> 💡 **오늘의 프레이밍**: 우리는 '경비행기'로 비행 원리를 배우는 중입니다. 여객기(GPT-4급)도 같은 원리로 납니다. 원리를 아는 사람만이 여객기를 제대로 조종(활용)할 수 있습니다.

## ⚠️ 자주 하는 실수

- **"GPT는 답을 알고 말한다"** → GPT는 다음 단어를 확률로 이어갈 뿐, 사실 여부를 검증하는 장치가 없습니다. 그럴듯한 거짓말(환각)이 나오는 구조적 이유입니다.
- **"Transformer = GPT"** → Transformer는 구조(설계도)이고, GPT·BERT는 그 설계도로 지은 서로 다른 건물입니다.
- **temperature를 높일수록 좋다고 착각** → 창의성이 필요 없는 작업(분류·추출)에서 높은 temperature는 오답률만 높입니다.

## 🚀 실무에서는?

- **Attention 시각화는 디버깅 도구**: 모델이 엉뚱한 판정을 할 때, 어느 단어를 쳐다봤는지 확인하며 원인을 추적합니다.
- **모델 선택의 기준**: 이해 태스크(분류·검색)는 BERT 계열, 생성 태스크(요약·챗봇)는 GPT 계열 — 구조 차이를 알면 선택이 빨라집니다.
- **비용 설계**: 파라미터 수 ↑ = 성능 ↑ = 비용·지연시간 ↑. "가장 큰 모델"이 아니라 "태스크에 맞는 최소 모델"을 찾는 것이 실무 감각입니다.

## 🧩 Check Point

**Q1. RNN 대비 Transformer의 결정적 장점 2가지는?**

1. 병렬 처리 가능 + 멀리 떨어진 단어 관계도 직접 연결
2. 모델 크기가 항상 더 작음 + 학습 데이터가 필요 없음
3. 한국어 전용 설계 + 무료 사용
4. 순서대로만 읽음 + 문장이 짧을수록 유리함

**Q2. Attention이 하는 일을 가장 잘 표현한 것은?**

1. 문장에서 오타를 찾아 교정한다
2. 각 단어를 처리할 때 **문장 내 다른 단어들 중 어디에 주목할지 가중치를 계산**한다
3. 문장을 무조건 앞에서 뒤로만 읽는다
4. 자주 나온 단어를 삭제한다

**Q3. GPT가 환각(그럴듯한 거짓말)을 만들 수밖에 없는 구조적 이유는?**

1. 인터넷 연결이 끊겨서
2. 학습 데이터가 전부 소설이라서
3. GPT의 본질이 사실 검증기가 아니라 **"다음에 올 확률이 높은 단어"를 뽑는 기계**라서 — 그럴듯함과 사실 여부는 별개이기 때문
4. 개발자가 일부러 거짓말을 넣어서

<details>
<summary> ---------- ✅ 정답 확인 ---------- </summary>

**Q1 → 1번.** 순차 처리의 병목을 없애 GPU 병렬 학습이 가능해졌고, attention으로 문장 처음과 끝도 한 번에 연결합니다. 이 두 가지가 "대형 모델 시대"를 열었습니다.

**Q2 → 2번.** "it"을 처리할 때 "glass"에 강하게 주목하는 heatmap, 직접 보셨죠 — 그 가중치 계산이 attention의 전부입니다.

**Q3 → 3번.** 수업에서 본 그대로, GPT는 매 순간 다음 토큰의 확률분포를 뽑을 뿐입니다. "확률적으로 자연스러운 문장"과 "사실인 문장"은 다른 문제라서, 모르는 것도 그럴듯하게 이어 씁니다.

</details>

## 💻 실습 (자습용)

In [ ]:
# TODO 1 (다음 단어 확률): "I want to eat" 다음에 올 단어 TOP 5를 확률과 함께 출력하세요


<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 예시 정답
inp = gpt2_tok("I want to eat", return_tensors="pt")
with torch.no_grad():
    lg = gpt2(**inp).logits
pb = torch.softmax(lg[0, -1], dim=-1)
tp = torch.topk(pb, 5)
for p, idx in zip(tp.values, tp.indices):
    print(f"'{gpt2_tok.decode(idx).strip()}'  {p.item()*100:.1f}%")
```

</details>

In [ ]:
# 코드 입력




In [ ]:
# TODO 2 (생성 실험): 좋아하는 영어 문장 앞부분으로 20토큰을 생성해보세요 (temperature 0.7)


<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 예시 정답
inp = gpt2_tok("On my first day at the company", return_tensors="pt")
out = gpt2.generate(**inp, max_new_tokens=20, do_sample=True,
                    temperature=0.7, pad_token_id=gpt2_tok.eos_token_id)
print(gpt2_tok.decode(out[0]))
```

</details>

In [ ]:
# 코드 입력




In [ ]:
# TODO 3 (attention 관찰): "she poured water into the glass because it was empty" 문장의
#        attention map을 그리고, 'it' 이 어떤 단어를 참조하는지 관찰하세요


<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 예시 정답
inp = tok("she poured water into the glass because it was empty", return_tensors="pt")
with torch.no_grad():
    out = model(**inp)
tks = tok.convert_ids_to_tokens(inp["input_ids"][0])
am = out.attentions[-1][0].mean(dim=0).numpy()
plt.figure(figsize=(7, 6))
plt.imshow(am, cmap="Blues")
plt.xticks(range(len(tks)), tks, rotation=45)
plt.yticks(range(len(tks)), tks)
plt.tight_layout(); plt.show()
# 'it' 행에서 'glass' 방향의 시선이 상대적으로 강한지 확인해보세요
```

</details>

In [ ]:
# 코드 입력




# 'it' 행에서 'glass' 방향의 시선이 상대적으로 강한지 확인해보세요

## 📝 실습 과제 (자습용)

In [ ]:
# 과제 1: 같은 프롬프트로 temperature 0.2 / 0.7 / 1.5 각각 3회씩 생성해보고,
#         temperature에 따라 결과의 '일관성'과 '다양성'이 어떻게 변하는지 표로 정리하세요

# 과제 2: GPT-2가 명백히 틀린 사실을 생성하는 사례를 1개 찾아 기록하고,
#         '다음 단어 예측' 구조 관점에서 왜 이런 일이 생기는지 3줄로 설명해보세요


<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 과제 1 예시 정답 (구조 예시)
prompt2 = "The future of AI is"
for temp in [0.2, 0.7, 1.5]:
    print(f"--- temperature {temp} ---")
    for _ in range(2):
        inp = gpt2_tok(prompt2, return_tensors="pt")
        out = gpt2.generate(**inp, max_new_tokens=15, do_sample=True,
                            temperature=temp, pad_token_id=gpt2_tok.eos_token_id)
        print(gpt2_tok.decode(out[0]))
    print()

# 과제 2 예시 정답
# 예: "The capital of Australia is Sydney..." 같은 출력이 흔히 발생
# 이유: GPT는 '학습 데이터에서 자주 함께 등장한 단어'를 이어갈 뿐,
#       사실 데이터베이스를 조회하지 않기 때문. Sydney가 Australia와
#       자주 동반 등장했다면 확률적으로 선택될 수 있음.
```

</details>

In [ ]:
# 과제 1 정답





In [ ]:
# 과제 2 정답





## 📌 핵심 정리

- Transformer = 순서대로 읽기(RNN)를 버리고 **문장 전체를 한 번에 + Attention으로 상호 참조**하는 구조. 병렬화 덕분에 LLM 시대가 열림
- **Attention** = 각 단어가 다른 단어를 얼마나 참고할지 학습하는 가중치 — heatmap으로 직접 확인 가능
- **GPT = 다음 단어 확률 예측의 반복.** 사실 검증 장치가 없어 환각이 구조적으로 발생
- GPT-2와 최신 모델의 차이는 원리가 아니라 **크기(스케일)** — 경비행기로 비행 원리를 배우는 중

---
# 📘 CH16-05. HuggingFace Pipeline — 3줄로 진짜 AI 쓰기 + 🥊 최종 대결

⏱️ **예상 소요시간**: 50분

> 🎬 **미션 ⑤ (최종)** — 팀장: "아침에 만든 규칙 기반 v1, 수고했지만 이제 은퇴시킵시다. **수백만 문장을 학습한 진짜 모델**로 교체 배치하세요. 그리고... 정말 더 나은지 **정면 대결로 증명**하세요."


## 🤔 먼저 생각해보기

- 감성 분석 모델을 직접 학습시키려면 라벨링된 문장 수십만 개와 GPU가 필요합니다. 우리에게 둘 다 없다면?
- 앱을 만들 때 지도 기능을 직접 개발하지 않고 지도 API를 갖다 쓰듯, AI 모델도 '갖다 쓰는' 생태계가 있지 않을까요?

## 📖 핵심 개념

### 1) HuggingFace — AI 모델계의 앱스토어

🔗 **https://huggingface.co** (HuggingFace — AI 모델계의 앱스토어)

**HuggingFace Hub**에는 전 세계 기업·연구자가 공개한 사전학습 모델이 100만 개 이상 올라와 있습니다. 감성 분석, 번역, 요약, 질의응답… 대부분 무료로 즉시 사용 가능합니다.

그리고 `pipeline` 함수는 그 모델들을 **단 3줄로** 쓰게 해주는 마법의 인터페이스입니다.

```
전통 방식: 데이터 수집 → 라벨링 → 모델 설계 → 학습 → 평가 → 배포   (수 주 ~ 수 개월)
pipeline:  from transformers import pipeline → 로딩 → 호출              (3줄, 30초)
```

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
from transformers import pipeline

# 단 3줄: 수백만 문장으로 학습된 감성 분석 모델 배치 완료
clf = pipeline("sentiment-analysis",
               model="distilbert-base-uncased-finetuned-sst-2-english")

print(clf("This song is a masterpiece!"))
print(clf("Worst title track ever, so disappointed."))
```

</details>

In [ ]:
# pipeline 학습된 AI 모델을 몇 줄만으로 바로 사용할 수 있게 만들어주는 함수
# pipeline("sentiment-analysis") 감성분석 AI
# pipeline("text-generation") 문장 생성 AI
# pipeline("translation") 번역 AI
# model="distilbert-base-uncased-finetuned-sst-2-english"





### 2) 팬 댓글 전체 자동 분석 — 진짜 리포트 만들기

아침의 영어 팬 댓글들을 새 모델로 일괄 분석하고, 팀장 보고용 차트까지 뽑아봅시다.

In [ ]:
import pandas as pd

# NEBULA 신곡 'Supernova' 발매 직후 수집된 팬 댓글 (실습용 가상 데이터)
fan_comments = [
    "이 노래 진짜 미쳤다ㅋㅋㅋ 벌써 100번 들음",
    "This song is a masterpiece, the chorus is stuck in my head!",
    "솔직히 이번 앨범은 좀 실망이야... 전작이 더 좋았어",
    "The music video is absolutely stunning, best comeback ever",
    "타이틀곡 별로다. 수록곡이 더 나은 듯",
    "NEBULA never disappoints, streaming all day!",
    "뮤비 미장센 무엇... 소름 돋았어 진심",
    "Honestly the autotune ruined it for me, disappointed",
    "스밍 준비 완료!! 오늘부터 1일 1스트리밍 간다",
    "I was not expecting much but wow, this exceeded everything",
    "노래는 좋은데 안무가 너무 아쉽다",
    "Worst title track they've ever released, skip",
    "기대 안 했는데 웬걸, 역대급이잖아?",
    "The bridge part gave me chills, pure art",
    "이게 노래냐... 3분이 아깝다",
    "Not bad at all, actually really impressive!",
    "멜로디 중독성 무엇 하루종일 흥얼거림",
    "Boring and generic, sounds like every other song",
    "컨셉 소화력 하나는 인정. 근데 노래는 글쎄",
    "Perfect song does not exi— oh wait, it just dropped",
]

df = pd.DataFrame({"comment": fan_comments})
print(f"수집된 댓글: {len(df)}개")
df.head(10)

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 영어 팬 댓글만 추출 (한국어 처리는 잠시 후 번역 파이프라인과 연계!)
en_comments = [c for c in fan_comments if not any('가' <= ch <= '힣' for ch in c)]

results = clf(en_comments)
for c, r in zip(en_comments, results):
    emoji = "😊" if r["label"] == "POSITIVE" else "😞"
    print(f"[{emoji} {r['label']:8s} {r['score']:.2f}] {c}")
```

</details>

In [ ]:
# 영어 팬 댓글만 추출 (한국어 처리는 잠시 후 번역 파이프라인과 연계!)






<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
import matplotlib.pyplot as plt

# 팀장 보고용 요약 차트
labels = [r["label"] for r in results]
pos = labels.count("POSITIVE")
neg = labels.count("NEGATIVE")

plt.figure(figsize=(6, 4))
plt.bar(["POSITIVE", "NEGATIVE"], [pos, neg], color=["#2E4BE8", "#E8930C"])
plt.title(f"'Supernova' Global Fan Sentiment  ({pos+neg} comments)")
for i, v in enumerate([pos, neg]):
    plt.text(i, v + 0.05, str(v), ha="center", fontsize=13)
plt.show()
print(f"긍정 비율: {pos/(pos+neg)*100:.0f}% → 컴백 반응 양호! 프로모션 유지 권고")
```

</details>

In [ ]:
import matplotlib.pyplot as plt

# 팀장 보고용 요약 차트





plt.figure(figsize=(6, 4))

plt.bar(["POSITIVE", "NEGATIVE"], [pos, neg], color=["#2E4BE8", "#E8930C"])
plt.title(f"'Supernova' Global Fan Sentiment  ({pos+neg} comments)")

for i, v in enumerate([pos, neg]):
    plt.text(i, v + 0.05, str(v), ha="center", fontsize=13)

plt.show()
print(f"긍정 비율: {pos/(pos+neg)*100:.0f}% → 컴백 반응 양호! 프로모션 유지 권고")

### 3) 파이프라인 갈아끼우기 — 번역기

`pipeline`의 진짜 매력: **태스크 이름과 모델만 바꾸면** 완전히 다른 AI가 됩니다. 한국어 팬 댓글을 본사 보고용 영어로 번역해봅시다.

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
translator = pipeline("translation", model="Helsinki-NLP/opus-mt-ko-en")

ko_comments = [c for c in fan_comments if any('가' <= ch <= '힣' for ch in c)][:5]

for c in ko_comments:
    en = translator(c)[0]["translation_text"]
    print(f"🇰🇷 {c}")
    print(f"🇺🇸 {en}\n")
```

</details>

In [ ]:
# model="Helsinki-NLP/opus-mt-ko-en

translator = pipeline("translation", model="Helsinki-NLP/opus-mt-ko-en")

ko_comments = [c for c in fan_comments if any('가' <= ch <= '힣' for ch in c)][:5]

for c in ko_comments:
    en = translator(c)[0]["translation_text"]
    print(f"🇰🇷 {c}")
    print(f"🇺🇸 {en}\n")

### 4) 파이프라인 조립 — 번역 + 감성 분석 체인

감성 분석 모델은 영어 전용이라 한국어 댓글을 못 읽습니다. 그렇다면? **번역 파이프라인과 이어 붙이면 됩니다.** 레고처럼 조립하는 것이 파이프라인 설계의 핵심 감각입니다.

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
def analyze_korean(comment):
    """한국어 댓글 → 영어 번역 → 감성 분석 (2단 파이프라인 체인)"""
    en = translator(comment)[0]["translation_text"]
    result = clf(en)[0]
    return en, result

for c in ko_comments:
    en, r = analyze_korean(c)
    emoji = "😊" if r["label"] == "POSITIVE" else "😞"
    print(f"[{emoji} {r['score']:.2f}] {c}")
    print(f"        └ 번역: {en}\n")
```

</details>

In [ ]:
def analyze_korean(comment):
    """한국어 댓글 → 영어 번역 → 감성 분석 (2단 파이프라인 체인)"""
    en = translator(comment)[0]["translation_text"]
    result = clf(en)[0]
    return en, result

for c in ko_comments:
    en, r = analyze_korean(c)
    emoji = "😊" if r["label"] == "POSITIVE" else "😞"
    print(f"[{emoji} {r['score']:.2f}] {c}")
    print(f"        └ 번역: {en}\n")

### 5) 태스크 파노라마 — pipeline으로 열리는 세계

오늘 쓴 것은 빙산의 일각입니다. 태스크 이름만 바꾸면:

| 태스크 이름 | 하는 일 | 예시 활용 |
|---|---|---|
| `sentiment-analysis` | 감성 분류 | 오늘의 미션 ✅ |
| `translation` | 번역 | 글로벌 팬 리포트 ✅ |
| `text-generation` | 텍스트 생성 | 답글 초안 작성 |
| `summarization` | 요약 | 긴 팬레터 세 줄 요약 |
| `question-answering` | 질의응답 | 공지문에서 콘서트 날짜 추출 |
| `zero-shot-classification` | 라벨 즉석 분류 | 문의를 배송/환불/칭찬으로 분류 |
| `ner` | 개체명 인식 | 문장에서 인명·지명·날짜 추출 |

> 💡 각 태스크의 기본 모델은 대부분 수백 MB~수 GB입니다. 실무에서는 필요한 태스크의 모델만 골라 배포합니다.

## 🥊 최종 대결 — 규칙 기반 v1 vs 사전학습 모델

아침에 만든 v1과 방금 배치한 DistilBERT의 정면 승부. 심판용 문제는 **v1을 괴롭혔던 함정 유형**(부정어 뒤집기·반어법·미묘한 표현)으로 구성한 10문장입니다.

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 심판용 데이터셋: (문장, 정답) — 부정어/반어법/미묘한 표현 함정 포함
battle_set = [
    ("Not bad at all, actually amazing!",                "긍정"),
    ("This is anything but boring",                      "긍정"),
    ("I can't stop listening to this song",              "긍정"),
    ("It could have been worse, honestly",               "부정"),
    ("The best part was when it ended",                  "부정"),
    ("Never expected it to be this good",                "긍정"),
    ("Not my cup of tea, sorry",                         "부정"),
    ("I don't hate it, I really don't",                  "긍정"),
    ("Perfect example of wasted potential",              "부정"),
    ("Nothing beats this masterpiece",                   "긍정"),
]

def v1_binary(text):
    r = sentiment_v1(text)
    return "긍정" if "긍정" in r else "부정"   # 대결에서는 중립도 부정으로 간주

def bert_binary(text):
    return "긍정" if clf(text)[0]["label"] == "POSITIVE" else "부정"

v1_score, bert_score = 0, 0
print(f"{'문장':45s} {'정답':4s} {'v1':4s} {'BERT':4s}")
print("-" * 70)
for text, answer in battle_set:
    p1, p2 = v1_binary(text), bert_binary(text)
    v1_score += (p1 == answer)
    bert_score += (p2 == answer)
    m1 = "✅" if p1 == answer else "❌"
    m2 = "✅" if p2 == answer else "❌"
    print(f"{text:45s} {answer}  {m1}{p1} {m2}{p2}")

print("-" * 70)
print(f"🏆 최종 스코어 — 규칙 기반 v1: {v1_score}/10  vs  DistilBERT: {bert_score}/10")
```

</details>

In [ ]:
# 심판용 데이터셋: (문장, 정답) — 부정어/반어법/미묘한 표현 함정 포함
battle_set = [
    ("Not bad at all, actually amazing!",                "긍정"),
    ("This is anything but boring",                      "긍정"),
    ("I can't stop listening to this song",              "긍정"),
    ("It could have been worse, honestly",               "부정"),
    ("The best part was when it ended",                  "부정"),
    ("Never expected it to be this good",                "긍정"),
    ("Not my cup of tea, sorry",                         "부정"),
    ("I don't hate it, I really don't",                  "긍정"),
    ("Perfect example of wasted potential",              "부정"),
    ("Nothing beats this masterpiece",                   "긍정"),
]

def v1_binary(text):
    r = sentiment_v1(text)
    return "긍정" if "긍정" in r else "부정"   # 대결에서는 중립도 부정으로 간주

def bert_binary(text):
    return "긍정" if clf(text)[0]["label"] == "POSITIVE" else "부정"

v1_score, bert_score = 0, 0
print(f"{'문장':45s} {'정답':4s} {'v1':4s} {'BERT':4s}")
print("-" * 70)
for text, answer in battle_set:
    p1, p2 = v1_binary(text), bert_binary(text)
    v1_score += (p1 == answer)
    bert_score += (p2 == answer)
    m1 = "✅" if p1 == answer else "❌"
    m2 = "✅" if p2 == answer else "❌"
    print(f"{text:45s} {answer}  {m1}{p1} {m2}{p2}")

print("-" * 70)
print(f"🏆 최종 스코어 — 규칙 기반 v1: {v1_score}/10  vs  DistilBERT: {bert_score}/10")

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 대결 결과 시각화
plt.figure(figsize=(6, 4))
bars = plt.bar(["Rule-based v1\n(오전에 직접 제작)", "DistilBERT\n(사전학습 모델)"],
               [v1_score, bert_score], color=["#E8930C", "#2E4BE8"])
plt.ylim(0, 10)
plt.ylabel("정답 수 (10문제)")
plt.title("🥊 Sentiment Battle: Rules vs Pretrained Model")
for bar, score in zip(bars, [v1_score, bert_score]):
    plt.text(bar.get_x() + bar.get_width()/2, score + 0.2, f"{score}/10",
             ha="center", fontsize=14, fontweight="bold")
plt.show()
```

</details>

In [ ]:
# 대결 결과 시각화
plt.figure(figsize=(6, 4))
bars = plt.bar(["Rule-based v1", "DistilBERT"],
               [v1_score, bert_score], color=["#E8930C", "#2E4BE8"])
plt.ylim(0, 10)
plt.ylabel("Answer")
plt.title("🥊 Sentiment Battle: Rules vs Pretrained Model")
for bar, score in zip(bars, [v1_score, bert_score]):
    plt.text(bar.get_x() + bar.get_width()/2, score + 0.2, f"{score}/10",
             ha="center", fontsize=14, fontweight="bold")
plt.show()

**대결 해설**

- v1이 무너진 문제는 대부분 **부정어·반어법** 유형 — 단어 사전으로는 맥락을 못 봅니다.
- DistilBERT는 수백만 문장에서 "not bad = 긍정" 같은 패턴을 통계적으로 흡수했기에 맥락을 읽습니다.
- 단, BERT도 만능은 아닙니다. "The best part was when it ended" 같은 고급 반어법은 최신 대형 모델도 종종 틀립니다. **모델은 확률 기계일 뿐, 검증은 사람의 몫**이라는 것이 오늘의 마지막 교훈입니다.

## ⚠️ 자주 하는 실수

- **모델 언어 확인 누락** → 영어 전용 모델에 한국어를 넣으면 에러 없이 '그럴듯한 오답'이 나옵니다. 가장 위험한 유형의 버그입니다.
- **score를 '정확도'로 오해** → pipeline의 score는 그 판정에 대한 확신도이지, 모델이 맞았다는 보장이 아닙니다.
- **첫 로딩이 느리다고 당황** → 모델 파일 로딩 때문이며 최초 1회뿐입니다. 실무 서비스는 모델을 메모리에 상주시켜 해결합니다.

## 🚀 실무에서는?

- **PoC(개념 검증)의 표준**: "이 기능 AI로 될까요?"라는 질문에, pipeline으로 반나절 만에 데모를 만들어 보여주는 것이 현업의 흔한 첫 수입니다.
- **파이프라인 체인 설계**: 오늘의 번역→감성분석처럼, 실제 서비스는 여러 모델을 이어 붙인 체인으로 구성됩니다 (음성인식→번역→요약 등).
- **모델 카드 읽기**: HuggingFace의 각 모델 페이지(모델 카드)에는 학습 데이터·언어·한계가 명시되어 있습니다. 배포 전 필독 문서입니다.

## 🧩 Check Point

1. 사전학습 모델을 쓰는 것이 직접 학습보다 유리한 상황은?
2. 번역→감성분석 체인을 만든 이유는? (어떤 문제를 우회했나요?)
3. 최종 대결에서 v1이 진 근본 원인을 '맥락'이라는 단어를 써서 설명하면?

## 💻 실습 (자습용)

In [ ]:
# TODO 1 (감성 분석): 영어 문장 3개를 직접 만들어 clf로 분석해보세요 (함정 문장 1개 포함)


<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 예시 정답
my_sentences = [
    "I absolutely love this new feature",
    "The service was slow and the staff was rude",
    "Well, that was not entirely terrible",   # 함정: 이중 부정
]
for s in my_sentences:
    print(clf(s)[0], "-", s)
```

</details>

In [ ]:
# 코드 입력




In [ ]:
# TODO 2 (번역 체인): 한국어 문장 "오늘 수업 진짜 유익했다" 를
#        analyze_korean 함수로 분석해보세요


<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 예시 정답
en, r = analyze_korean("오늘 수업 진짜 유익했다")
print(f"번역: {en}")
print(f"판정: {r}")
```

</details>

In [ ]:
# 코드 입력




In [ ]:
# TODO 3 (새 태스크 개척): text-generation 파이프라인(gpt2)으로
#        "Thank you for listening to our new song," 뒤를 이어 팬 감사 메시지 초안을 생성해보세요


<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 예시 정답
gen = pipeline("text-generation", model="gpt2")
draft = gen("Thank you for listening to our new song,",
            max_new_tokens=30, do_sample=True, temperature=0.8)
print(draft[0]["generated_text"])
# 품질이 어색해도 괜찮습니다 — 초안 생성 후 사람이 다듬는 것이 실무 워크플로우입니다
```

</details>

In [ ]:
# 코드 입력



# 품질이 어색해도 괜찮습니다 — 초안 생성 후 사람이 다듬는 것이 실무 워크플로우입니다

## 📝 실습 과제 (자습용)

In [ ]:
# 과제 1: battle_set에 여러분이 설계한 함정 문장 5개를 추가해서 대결을 다시 진행하고,
#         BERT까지 틀리게 만드는 문장을 1개 이상 찾아보세요

# 과제 2: zero-shot-classification 파이프라인으로 아래 고객 문의 3건을
#         ["배송", "환불", "칭찬"] 라벨로 분류해보세요
#         문의: "When will my album arrive?", "I want my money back",
#               "Best fan merchandise ever!"
#         (모델: facebook/bart-large-mnli — 약 1.6GB, 자습 환경에서 다운로드)


<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 과제 1 예시 정답 (추가 함정 예시)
extra_traps = [
    ("Yeah right, like that's going to work", "부정"),   # 빈정거림
    ("I'm not saying it's bad, but...", "부정"),          # 완곡한 부정
    ("This slaps so hard", "긍정"),                        # 슬랭
]
for text, answer in extra_traps:
    print(f"정답 {answer} | v1: {v1_binary(text)} | BERT: {bert_binary(text)} | {text}")

# 과제 2 예시 정답 (자습 환경에서 실행)
# zsc = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
# inquiries = ["When will my album arrive?", "I want my money back",
#              "Best fan merchandise ever!"]
# for q in inquiries:
#     r = zsc(q, candidate_labels=["shipping", "refund", "compliment"])
#     print(f"{q} → {r['labels'][0]} ({r['scores'][0]:.2f})")
```

</details>

In [ ]:
# 과제 1 예시 정답 (추가 함정 예시)






In [ ]:
# 과제 2 예시 정답 (자습 환경에서 실행)

# zsc = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
# inquiries = ["When will my album arrive?", "I want my money back",
#              "Best fan merchandise ever!"]
# for q in inquiries:
#     r = zsc(q, candidate_labels=["shipping", "refund", "compliment"])
#     print(f"{q} → {r['labels'][0]} ({r['scores'][0]:.2f})")

## 📌 핵심 정리

- **HuggingFace Hub** = 100만 개 이상의 공개 사전학습 모델 저장소, `pipeline` = 3줄 사용 인터페이스
- 태스크 이름만 바꾸면 감성분석·번역·요약·QA 등으로 즉시 전환 — **모델을 '만드는' 시대에서 '고르고 조립하는' 시대로**
- 파이프라인은 레고처럼 **체인 조립** 가능 (번역→감성분석)
- 최종 대결: 사전학습 모델은 **맥락**을 읽어 규칙 기반을 압도하지만, 여전히 확률 기계 — 검증은 사람의 몫

---
# 🌟 오늘 하루 총정리 — 스텔라 엔터 인턴 일지

| 미션 | 배운 것 | 핵심 한 줄 |
|---|---|---|
| ① NLP 첫 만남 | 규칙 기반 감성 분석 | 글자를 의미 있는 숫자로 바꾸는 것이 NLP의 전부 |
| ② 토큰화 | split → nltk → subword | 자르는 방식이 성능과 요금을 결정한다 |
| ③ 임베딩 | GloVe, 단어 연금술, 단어 지도 | 의미가 비슷하면 좌표가 가깝다 |
| ④ Transformer/GPT | Attention, 다음 단어 예측 | LLM = 문맥을 참조하는 확률 기계 |
| ⑤ Pipeline | 3줄 배치, 체인 조립, 최종 대결 | 만들지 말고 골라서 조립하라 |

**오늘의 큰 그림**: `글자 → 토큰 → 임베딩(좌표) → Attention(문맥) → 예측/생성` — 이 다섯 단계가 ChatGPT를 포함한 모든 현대 언어 AI의 공통 골격입니다.

---
# 🎁 보너스 (자습) — 자유 실험실

수업이 끝난 뒤 놀아보세요. 제출용이 아니라 놀이용입니다.

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 놀이 1: GPT-2 릴레이 소설 — 문장을 하나 쓰고, GPT-2가 이어 쓰게 한 뒤,
#         그 결과에 다시 이어 쓰게 하는 것을 3회 반복해보세요. 어디까지 산으로 가는지 관찰!

story = "Once upon a time, a small AI woke up in a laptop"
for round_num in range(3):
    inp = gpt2_tok(story, return_tensors="pt")
    out = gpt2.generate(**inp, max_new_tokens=20, do_sample=True,
                        temperature=0.9, pad_token_id=gpt2_tok.eos_token_id)
    story = gpt2_tok.decode(out[0])
    print(f"--- Round {round_num + 1} ---")
    print(story, "\n")
```

</details>

In [ ]:
# 놀이 1: GPT-2 릴레이 소설 — 문장을 하나 쓰고, GPT-2가 이어 쓰게 한 뒤,
#         그 결과에 다시 이어 쓰게 하는 것을 3회 반복해보세요. 어디까지 산으로 가는지 관찰!

story = "Once upon a time, a small AI woke up in a laptop"





<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 놀이 2: 무엇이든 판독기 — 친구에게 받은 문자, 좋아하는 영화 대사(영어),
#         아무 문장이나 넣고 감성 판정을 확인해보세요. 모델을 속이는 데 성공하면 인증!

print(clf("I love deadlines. I love the whooshing noise they make as they go by."))
```

</details>

In [ ]:
# 놀이 2: 무엇이든 판독기 — 친구에게 받은 문자, 좋아하는 영화 대사(영어),
#         아무 문장이나 넣고 감성 판정을 확인해보세요. 모델을 속이는 데 성공하면 인증!

print(clf("I love deadlines. I love the whooshing noise they make as they go by."))